[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_3_Nonlinear_Charpit_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


# 📡 اللاب 3: المعادلات غير الخطية وطريقة شاربيت لموجات الرادار
### *غرفة التحكم: حساس الرادار البصري وموجات Eikonal*
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)  
**موضوع المنهج:** الباب الأول، الأقسام 1.3.2–1.3.6 (صفحات 17–28)

---
### 🧭 حجر رشيد (The Rosetta Stone): ترجمة الرياضيات إلى لغة البشر وكود بايثون
| 1. معادلات السبورة في المحاضرة | 2. المعنى الفيزيائي البسيط | 3. كود بايثون المقابل |
| :--- | :--- | :--- |
| $p^2 + q^2 = 1$ | "معادلة Eikonal: موجات الرادار تنتشر بسرعة ثابتة في الفضاء." | `np.hypot(X - x0, Y - y0)` |
| $z = p x + q y + f(p, q)$ | "صيغة كليروت (Clairaut Form): حلها التام خطوط مستقيمة." | `z = a*x + b*y + f(a, b)` |
| $(x - x_0)^2 + (y - y_0)^2 = z^2$ | "الغلاف الدائري لموجة الرادار المتوسعة." | `dist_grid = np.sqrt(dx**2 + dy**2)` |

---
## 1. المعنى البسيط: "دوائر الماء المتوسعة"
عندما تلقي حجراً في بركة ماء هادئة، تتسع دوائر الأمواج بسرعة ثابتة. المسافة المقطوعة تعتمد تربيعياً على الميول:
$$\|\nabla T\|^2 = p^2 + q^2 = 1$$
هذه المعادلة غير خطية لأن المشتقتين $p$ و $q$ مرفوعتان للأس 2.
* **طريقة شاربيت (Charpit's Method):** هي الطريقة الشاملة لحل أي معادلة غير خطية من الرتبة الأولى عبر تتبع إحداثيات الموقع والميل معاً في فضاء خماسي الأبعاد $(x, y, z, p, q)$!

---
## 2. ما فائدة هذه الطريقة ولماذا نقوم بها؟ (The "WHY")
* **لماذا نقوم بها؟** أجهزة الاستشعار بالرادار والليزر (LIDAR) على المريخ تحسب مسافة الصخور بقياس زمن ارتداد الموجة الضوئية.
* **ماذا تفيد مركبة المريخ؟** تمكن المركبة من بناء خريطة مسافات دائرية متوسعة حول العوائق لتفادي الاصطدام بها في التضاريس المجهولة.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab3

%matplotlib inline
print("تم تهيئة بيئة اللاب 3 بنجاح! جاهز لحساب موجات الرادار.")


---
## 3. الوصفة الرياضية والأشكال القياسية الأربعة (Standard Forms 1–4)

1. **Form 1 ($f(p, q) = 0$):** تحتوي على $p, q$ فقط $\implies$ نضع $p = a$ ونحل لإيجاد $q = b$. التكامل التام: $z = ax + by + c$.
2. **Form 2 ($f(z, p, q) = 0$):** تحتوي على $z, p, q$ فقط $\implies$ نضع $u = x + ay$.
3. **Form 3 ($f(x, p) = g(y, q)$):** قابلة للفصل $\implies$ نساوي الطرفين بثابت $a$.
4. **Form 4 (Clairaut Form: $z = px + qy + f(p, q)$):** الحل التام المباشر هو استبدال $p \to a$ و $q \to b$:
   $$z = ax + by + f(a, b)$$


---
## 4. 📝 تحدي الورقة والقلم 1 (صيغة كليروت من المنهج)

> 💡 *(إذا واجهتك صعوبة، راجع `INSTRUCTOR_ANSWER_KEY.md` القسم 3.1).*

**المسألة:** أوجد التكامل التام لمعادلة كليروت:
$$z = p x + q y + p q$$
بما أنها على صيغة كليروت، ضع $p = a$ و $q = b$. اكتب الحل التام أدناه:


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ###
student_clairaut = "z = ax + by + ab"
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab3.check_paper_challenge_1(student_clairaut)


---
## 5. 🚀 تحدي جسر مركبة المريخ (غلاف موجة الرادار Eikonal)

معادلة رادار المسح الضوئي للمركبة هي:
$$p^2 + q^2 = 1$$

**على ورقتك:**
1. باستخدام Form 1: ضع $p = a \implies q = \pm\sqrt{1 - a^2}$.
2. التكامل التام: $z = a x \pm\sqrt{1 - a^2} y + c$.
3. اشتق بالنسبة للبارامتر $a$ لمساواة المشتقة بالصفر وإيجاد الغلاف المفرد (Singular Envelope):
   $$\frac{\partial z}{\partial a} = x \pm \frac{-a}{\sqrt{1 - a^2}} y = 0 \implies x^2 + y^2 = z^2$$

اكتب معادلة الغلاف الدائري المتوسع أدناه:


In [ ]:
# خلية تقييم: غلاف موجة الرادار
### START CODE HERE ###
student_envelope = "x^2 + y^2 = z^2"
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab3.check_paper_challenge_2(student_envelope)


---
## 6. 💻 التطبيق البرمجي ببايثون: حساب خريطة مسافات الرادار

سنكتب دالة تحسب خريطة المسافات الإقليدية المتوسعة من مصدر الرادار محققة معادلة $\|\nabla T\| = 1$:


In [ ]:
# دالة تقييم برمجية: compute_eikonal_wavefront
def compute_eikonal_wavefront(grid_size=(50, 50), source=(25, 25)):
    # تحسب خريطة زمن وصول الموجة الدائرية T(x, y) من موقع الرادار
    Ny, Nx = grid_size
    Y, X = np.ogrid[:Ny, :Nx]
    
    ### START CODE HERE ###
    dist = np.hypot(X - source[1], Y - source[0])
    ### END CODE HERE ###
    
    return dist

# تشغيل الاختبار الآلي:
test_lab3.test_eikonal_wavefront_distance(compute_eikonal_wavefront)


---
## 7. 🎨 الرؤية البصرية للفيزياء: موجات الرادار الدائرية
شغّل الخلية لرؤية خطوط انتشار جبهة الموجة الدائرية من الرادار:


In [ ]:
wavefront = compute_eikonal_wavefront((60, 60), source=(30, 30))

plt.figure(figsize=(8, 6))
cs = plt.contourf(wavefront, levels=20, cmap='plasma')
plt.colorbar(cs, label="Wavefront Arrival Distance T(x, y)")
plt.plot(30, 30, 'wo', markersize=10, label="Radar Beacon Source")
plt.title("Eikonal Radar Wavefront Propagation: $\|\nabla T\|^2 = 1$", fontsize=12)
plt.legend()
plt.show()

print("تهانينا! لقد أتممت اللاب 3 وأتقنت المعادلات غير الخطية وموجات Eikonal.")
